# Quantitative Universal Approximation for Noisy Quantum Neural Networks
**ArXivist-generated reproduction notebook**

Paper: [arXiv:2604.02064](https://arxiv.org/abs/2604.02064) — Gonon, Jacquier, Mordarski
Generated: 2026-07-22

This notebook walks through the key components of the implementation — the parameterized
quantum circuit, the depolarising-noise error bounds, and the Theorem 3.15 bias-cancellation
layer — runs a small-scale training loop on synthetic Black-Scholes data, and verifies that
the setup matches the paper's reported behavior on a mini-dataset.

> ⚠️ This paper's circuits are **small (3-8 qubits) and CPU-feasible** — no GPU is required,
> unlike most ArXivist-generated notebooks. The environment check below reflects that.


In [ ]:
# Check Python version, available compute, and key dependencies.
# NOTE: unlike most ML papers, this repo does NOT require a GPU -- circuits are
# 3-8 qubits and classical optimisation of the few circuit angles theta is CPU-feasible.
import sys

print(f"Python: {sys.version}")

try:
    import numpy
    print(f"NumPy: {numpy.__version__}")
except ImportError as e:
    print(f"[WARN] NumPy not found: {e}")

try:
    import scipy
    print(f"SciPy: {scipy.__version__}")
except ImportError as e:
    print(f"[WARN] SciPy not found: {e}")

try:
    import qiskit
    print(f"Qiskit: {qiskit.__version__}")
except ImportError as e:
    print(f"[WARN] Qiskit not found -- run the installation cell below: {e}")

try:
    import qiskit_aer
    print(f"Qiskit Aer: {qiskit_aer.__version__}")
except ImportError as e:
    print(f"[WARN] Qiskit Aer not found -- run the installation cell below: {e}")

try:
    import torch
    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()} (not required for this repo)")
except ImportError as e:
    print(f"[WARN] PyTorch not found -- run the installation cell below: {e}")

device = "cpu"  # this repo's circuits (3-8 qubits) are CPU-feasible; GPU is never required
print(f"\nUsing device: {device}")


In [ ]:
# Install the project in editable mode (run once).
import subprocess, sys, os

try:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", ".."],
        capture_output=True, text=True, cwd=os.getcwd(),
    )
    print(result.stdout[-2000:] if result.returncode == 0 else result.stderr[-2000:])
except Exception as e:
    print(f"[WARN] pip install -e .. failed ({e}); falling back to a relative sys.path insert.")

# Defensive fallback: also add ../src directly to sys.path, in case the editable
# install above didn't take (e.g. restrictive/managed Python environments).
src_path = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
if src_path not in sys.path:
    sys.path.insert(0, src_path)

try:
    import noisy_qnn_uat  # noqa: F401
    print("noisy_qnn_uat is importable.")
except ImportError as e:
    print(f"[ERROR] noisy_qnn_uat still not importable after install + fallback: {e}")


## Paper Overview

**Problem.** Parameterized quantum circuits (QNNs) can approximate a wide class of functions,
but nearly all existing universal-approximation theorems assume *noiseless* circuits -- an
assumption current NISQ hardware cannot satisfy. This paper asks: **which functions remain
representable, with what accuracy, once realistic hardware noise is introduced?**

**Core idea.** The authors:
1. Extend an existing noiseless universal-approximation theorem to *expectation* functions
   `f(x) = E[Phi(x+L)]` -- exactly the form of option prices in quantitative finance -- with
   sharp constants for exponential Levy models (Black-Scholes as the worked example).
2. Recast the QNN in the density-operator (CPTP channel) formalism and prove a fidelity-based
   error bound that holds for *any* quantum noise channel (Theorem 3.6).
3. Specialise to depolarising noise calibrated to real hardware, decomposing the total error
   into **statistical + systematic + offset + readout** terms (Theorem 3.17, Proposition 3.20),
   and show a **two-parameter affine post-processing layer exactly cancels the depolarising
   bias** (Theorem 3.15) -- a training-time fix requiring no circuit modification.
4. Validate everything by training Black-Scholes Put-pricing QNNs and executing them on the
   real IBM `ibm_fez` (Heron r2) processor: empirical error stayed inside the analytical
   envelope on 10/10 test points.

**Mapping to this repo:**

| Paper section | This repo |
|---|---|
| Sec. 3.2, 4.1 -- circuit V + U(theta,x) | `src/noisy_qnn_uat/models/qnn_circuit.py` |
| Eq. (2.2), Sec. 3.3, 4.1 -- measurement | `src/noisy_qnn_uat/models/measurement.py` |
| Sec. 3.5-3.7 -- depolarising noise, hardware calibration | `src/noisy_qnn_uat/models/noise_channels.py` |
| Theorem 3.15 -- affine bias cancellation | `src/noisy_qnn_uat/models/postprocessing.py` |
| Sec. 4.1 -- Methods A/B/C optimisation | `src/noisy_qnn_uat/training/trainer.py` |
| Statement 2.1, Thm 3.6/3.17, Prop 3.20 -- error bounds | `src/noisy_qnn_uat/evaluation/hardware_bounds.py` |
| Sec. 4.5 -- ibm_fez hardware execution | `scripts/run_hardware.py` |


## Component 1: The Parameterized Quantum Circuit (V + U(theta,x))

The circuit has three stages (Section 4.1): **state preparation V** (Hadamards on the
control qubits), the **parameterized unitary U(theta,x)** (a block-diagonal Fourier-like
ansatz, decomposed here via Uniformly Controlled Rotations), and **measurement**.

$$
U(\theta, x) = \sum_{k=0}^{n-1} |k\rangle\langle k| \otimes U^{(k)}(\theta_k, x), \qquad
\theta_k = (a_k, b_k, \gamma_k)
$$

The closed-form output this circuit targets (Section 4.1.1, used here to validate the
circuit) is:

$$
f^R_{n,\theta}(x) = \frac{1}{n}\sum_{i=1}^{n} R \cos(\gamma_i)\cos(b_i + a_i \cdot x)
$$


In [ ]:
import numpy as np

try:
    from noisy_qnn_uat.models.qnn_circuit import QNNCircuitBuilder, closed_form_reference_output

    n_accuracy_blocks, n_qubits, d, R = 4, 4, 2, 10.0
    builder = QNNCircuitBuilder(n_accuracy_blocks=n_accuracy_blocks, n_qubits=n_qubits)

    rng = np.random.default_rng(0)
    theta = [(rng.uniform(-1, 1, d), rng.uniform(-1, 1), rng.uniform(-1, 1))
             for _ in range(n_accuracy_blocks)]
    x = rng.uniform(0, 1, d)

    circuit = builder.assemble_circuit(theta, x)
    print(f"Circuit: {circuit.num_qubits} qubits, {len(circuit.data)} instructions")
    print(circuit.draw(output="text"))

    reference = closed_form_reference_output(theta, x, R, n_accuracy_blocks)
    print(f"\nClosed-form reference output f^R_{{n,theta}}(x) = {reference:.4f}")
except Exception as e:
    print(f"[ERROR] Circuit construction failed: {e}")


## Component 2: Measurement -- from raw counts to the scalar QNN output

Raw computational-basis outcomes are grouped by `outcome mod 4` into four probabilities
`P0..P3` (Section 4.1, "Measurement"), and the scalar QNN output follows Eq. (2.2):

$$
f^{R}_{n,\theta}(x) = R\left[1-2(P_1+P_2)\right]
$$

We validate the circuit above by actually sampling it on Qiskit's `AerSimulator` and
comparing against the closed-form reference (Section 4.1.1's own validation check --
residuals should stay within `R/sqrt(N_shots)`).


In [ ]:
try:
    from qiskit import transpile
    from qiskit_aer import AerSimulator
    from noisy_qnn_uat.models.measurement import MeasurementProcessor

    shots = 8192
    sim = AerSimulator()
    processor = MeasurementProcessor()

    transpiled = transpile(circuit, sim, optimization_level=1)
    counts = sim.run(transpiled, shots=shots).result().get_counts()
    probs = processor.group_counts(counts, n_accuracy_blocks, n_qubits)
    sampled_output = processor.qnn_output(probs, R)

    bound = R / np.sqrt(shots)
    residual = abs(sampled_output - reference)
    status = "within bound" if residual <= 3 * bound else "OUTSIDE bound (investigate!)"

    print(f"Grouped probabilities P0..P3: {probs}")
    print(f"Sampled circuit output:  {sampled_output:.4f}")
    print(f"Closed-form reference:   {reference:.4f}")
    print(f"Residual: {residual:.4f}  (paper's R/sqrt(N_shots) bound: {bound:.4f}) -> {status}")
except Exception as e:
    print(f"[ERROR] Measurement validation failed: {e}")


## Component 3: Depolarising Noise & Hardware Calibration (Sections 3.5-3.7)

Depolarising noise is modelled as $\Delta_\lambda(\rho) = (1-\lambda)\rho + \frac{\lambda}{d} I$.
Effective $\lambda_V, \lambda_U$ are computed directly from device calibration data
(single/two-qubit gate errors, $T_1$, $T_2$), giving the hardware fidelity factor:

$$
\alpha = (1-\lambda_V)(1-\lambda_U)
$$

Below we compute $\alpha$ using the paper's own reported `ibm_fez` (Heron r2) calibration
values (Appendix A, Table 1).


In [ ]:
try:
    from noisy_qnn_uat.models.noise_channels import HardwareNoiseCalibrator

    # ibm_fez calibration values, Appendix A Table 1
    eps_1q, eps_2q, t1_us, t2_us, t2q_ns = 2.761e-4, 2.548e-3, 144.97, 99.9, 68

    calibrator = HardwareNoiseCalibrator()
    lambda_v = calibrator.compute_lambda_V(eps_1q, n_qubits=5)
    _, n2q_ucr = calibrator.naive_and_ucr_two_qubit_gate_counts(n_accuracy_blocks=8, n_qubits=5)
    lambda_u = calibrator.compute_lambda_U(eps_2q, n2q_ucr, t1_us, t2_us, t2q_ns)
    alpha = calibrator.compute_alpha(lambda_v, lambda_u)

    print(f"lambda_V = {lambda_v:.6f}")
    print(f"lambda_U = {lambda_u:.6f}")
    print(f"alpha    = {alpha:.4f}   (hardware fidelity factor, Eq. 3.7)")
except Exception as e:
    print(f"[ERROR] Hardware noise calibration failed: {e}")


## Component 4: Theorem 3.15 -- Exact Depolarising-Bias Cancellation

A two-parameter affine post-processing layer can **exactly** cancel the depolarising bias:

$$
\tilde f^{R}_{n,\bar\theta}(x) = \beta_1 \tilde f^{R}_{n,\theta}(x) + \beta_2, \qquad
\beta_1 = \frac{1}{\alpha}, \quad
\beta_2 = -\beta_1 R (1-\alpha)\left(1-\frac{4n}{2^{\mathfrak{n}}}\right)
$$

We verify this numerically: apply noise to a noiseless output, then apply the closed-form
correction, and confirm we get back exactly the noiseless value.


In [ ]:
try:
    from noisy_qnn_uat.models.postprocessing import AffineNoiseCancellation

    R_val, n_blocks_val, n_qubits_val = 12.0, 8, 5
    f_noiseless = 5.3

    # Simulate the depolarising-noise contraction (Corollary 3.13)
    offset_term = 1.0 - (4 * n_blocks_val) / (2 ** n_qubits_val)
    f_noisy = alpha * f_noiseless + R_val * (1 - alpha) * offset_term

    beta1, beta2 = AffineNoiseCancellation.closed_form_correction(alpha, R_val, n_blocks_val, n_qubits_val)
    f_corrected = beta1 * f_noisy + beta2

    print(f"Noiseless output:        {f_noiseless:.6f}")
    print(f"Noisy output (alpha={alpha:.4f}): {f_noisy:.6f}")
    print(f"beta1={beta1:.4f}, beta2={beta2:.4f}")
    print(f"Corrected output:       {f_corrected:.6f}")
    print(f"Exact recovery: {abs(f_corrected - f_noiseless) < 1e-9}")
except Exception as e:
    print(f"[ERROR] Affine correction demo failed: {e}")


## Component 5: The Error Bounds (Statement 2.1, Theorem 3.17, Proposition 3.20)

The full hardware-validated bound (Eq. 4.4) decomposes into four terms:

$$
\varepsilon_{total} = \underbrace{\frac{\alpha L^1[\hat f]}{\sqrt n}}_{\text{statistical}}
+ \underbrace{(1-\alpha)\|f\|_{L^2(\mu)}}_{\text{systematic}}
+ \underbrace{R(1-\alpha)\left(1-\frac{4n}{2^{\mathfrak n}}\right)}_{\text{offset}}
+ \underbrace{4Rp}_{\text{readout}}
$$


In [ ]:
try:
    from noisy_qnn_uat.evaluation.hardware_bounds import ErrorBoundCalculator

    bound_calc = ErrorBoundCalculator()
    l1_fhat_example = 2.316   # Section 2.3.3 worked example (K=1, K_bar=0.4, sigma=0.2, T=1)
    f_l2_norm_estimate = 5.0  # illustrative -- paper does not give an exact value
    readout_p = 0.01          # ASSUMED (see configs/config.yaml hardware.readout_p_comment)

    decomposition = bound_calc.decompose_total_bound(
        alpha, l1_fhat_example, n_blocks_val, f_l2_norm_estimate, R_val, n_qubits_val, readout_p
    )
    for term, value in decomposition.items():
        print(f"  {term:>11}: {value:.4f}")
except Exception as e:
    print(f"[ERROR] Bound decomposition failed: {e}")


## Mini-Training Demonstration

We now fit circuit parameters `theta` on a **tiny synthetic** Black-Scholes Put dataset
(no downloads needed -- the data is generated analytically), using a short inline Adam
loop over the differentiable closed-form QNN output (see `training/trainer.py`'s
`fit_method_c_adam` for the full version used by `train.py`).

### Step 1: Generate a tiny synthetic dataset


In [ ]:
try:
    from noisy_qnn_uat.data.dataset import BlackScholesPutDataset
    from noisy_qnn_uat.data.transforms import InputNormalizer

    dataset = BlackScholesPutDataset()
    ranges = {
        "S_range": (0.8, 1.2), "K_range": (0.9, 1.1), "T_range": (0.5, 1.0),
        "r_range": (0.02, 0.05), "sigma_range": (0.1, 0.3),
        "n_samples": 40, "seed": 0,
    }
    x_raw, y_train = dataset.sample_training_grid(ranges)

    normalizer = InputNormalizer()
    x_min = np.array([0.8, 0.9, 0.5, 0.02, 0.1])
    x_max = np.array([1.2, 1.1, 1.0, 0.05, 0.3])
    x_train = normalizer.normalize(x_raw, x_min, x_max)

    R_train = float(np.ceil(1.1 * np.max(y_train)))
    print(f"Generated {x_train.shape[0]} synthetic (S,K,T,r,sigma) -> Put price samples")
    print(f"x_train shape: {x_train.shape}, y_train shape: {y_train.shape}, R={R_train}")
except Exception as e:
    print(f"[ERROR] Dataset generation failed: {e}")


### Step 2: Build the model (small circuit) and count parameters

In [ ]:
try:
    n_blocks_demo = 3  # small for a fast demo (paper uses 8 for the full experiment)
    n_params_per_block = x_train.shape[1] + 2  # (a_k vector of length d) + b_k + gamma_k
    n_total_params = n_blocks_demo * n_params_per_block
    print(f"Mini model: {n_blocks_demo} accuracy blocks, {n_total_params} trainable parameters "
          f"(a_k in R^{x_train.shape[1]}, b_k, gamma_k per block)")
except Exception as e:
    print(f"[ERROR] Model setup failed: {e}")


### Step 3: Train for a few steps with Adam, printing loss each step

In [ ]:
try:
    import torch

    rng_t = np.random.default_rng(1)
    a = torch.tensor(rng_t.uniform(-1, 1, (n_blocks_demo, x_train.shape[1])), requires_grad=True)
    b = torch.tensor(rng_t.uniform(-1, 1, n_blocks_demo), requires_grad=True)
    gamma = torch.tensor(rng_t.uniform(-1, 1, n_blocks_demo), requires_grad=True)

    x_t = torch.tensor(x_train, dtype=torch.float64)
    y_t = torch.tensor(y_train, dtype=torch.float64)
    optimizer = torch.optim.Adam([a, b, gamma], lr=0.05)

    n_steps = 8
    for step in range(n_steps):
        optimizer.zero_grad()
        phase = b.unsqueeze(1) + x_t @ a.T             # [n_train, n_blocks_demo]
        terms = torch.cos(gamma).unsqueeze(0) * torch.cos(phase)
        preds = (R_train / n_blocks_demo) * terms.sum(dim=1)
        loss = torch.mean((preds - y_t) ** 2)
        loss.backward()
        optimizer.step()
        print(f"  step {step}: loss = {loss.item():.4f}")
except Exception as e:
    print(f"[ERROR] Mini-training loop failed: {e}")


### Step 4: Check results -- loss decreasing, output shapes correct

In [ ]:
try:
    with torch.no_grad():
        phase = b.unsqueeze(1) + x_t @ a.T
        terms = torch.cos(gamma).unsqueeze(0) * torch.cos(phase)
        final_preds = (R_train / n_blocks_demo) * terms.sum(dim=1)

    print(f"Final predictions shape: {final_preds.shape} (expected: [{x_train.shape[0]}])")
    print(f"Final training loss: {loss.item():.4f}  (started at step 0; check it decreased above)")
    print(f"Final MAE: {torch.mean(torch.abs(final_preds - y_t)).item():.4f}")
except Exception as e:
    print(f"[ERROR] Results check failed: {e}")


## Paper Results Comparison

This mini-training run is illustrative only (3 accuracy blocks, 40 training points, 8 Adam
steps). The paper's actual reported results (from the SIR, `evaluation_protocol.reported_results`)
use `n_accuracy_blocks=8`, a 1600-point training grid, and real IBM `ibm_fez` hardware:


In [ ]:
paper_results = {
    "dataset": "Black-Scholes Put, hardware execution on ibm_fez",
    "metric": "Mean Absolute Error (MAE)",
    "reported_value": 2.345,
    "total_error_bound_reported": 18.578,
    "within_bound": "10/10 test points",
    "pearson_correlation_vs_comprehensive_noise_model": 0.9973,
    "hardware_fidelity_factor_alpha": 0.365,
    "baseline": "Classical analytical Black-Scholes formula",
}
print("Paper's claimed results (Section 4.5, Figure 4.2):")
for k, v in paper_results.items():
    print(f"  {k}: {v}")

print("\nTo reproduce these results at full scale, run:")
print("  python train.py --config configs/config.yaml --method A")
print("  python scripts/run_hardware.py --config configs/config.yaml --checkpoint <checkpoint>")
print("Then feed your results back to ArXivist's Results Comparator (Stage 6).")


## What to do next

1. **Full training**: `python train.py --config configs/config.yaml --method A` (or `B`/`C`)
2. **Evaluation**: `python evaluate.py --config configs/config.yaml --checkpoint checkpoints/theta_methodA_seed42.json`
3. **Hardware validation**: `python scripts/run_hardware.py --config configs/config.yaml --checkpoint <checkpoint>`
4. **Compare results**: feed your outputs back to ArXivist's Results Comparator (Stage 6)

**Implementation notes from the SIR** (top assumptions to verify before trusting exact numbers):

| Assumption | Confidence |
|---|---|
| Optimizer hyperparameters (Adam lr/betas, iteration budget) for Methods A/B/C are not stated in the paper; PyTorch defaults assumed | 0.3 |
| Readout error probability `p` used in the reported hardware bound is missing from the paper's own Appendix A hardware table; a placeholder value is used | 0.35 |
| `n0` padding value (`2^n = 4n` simplification, Remark 3.14) assumed for the main numerical experiments | 0.4 |

Full detail: `sir-registry/arxiv_2604_002064/sir.json` -> `implementation_assumptions`, `ambiguities`.
